# Notebook 02: Ticker Mapping and Source Resolution

## Academic Introduction

Ticker and security-identifier mapping is a critical step in event-study research because historical price extraction depends on market identifiers rather than company names alone. Company names may be written differently across the raw Excel file, the Tel Aviv Stock Exchange, Maya disclosures, Yahoo Finance, Google Finance, Bizportal, Investing.com, and news sources. In the Israeli market, a single company may require several identifiers: a local exchange symbol, a Yahoo Finance ticker, a Google Finance / Google Sheets symbol, a TASE security number, and possibly an ISIN.

Mapping uncertainty must be documented rather than hidden. Incorrect identifiers can contaminate return calculations, while missing identifiers can reduce the usable event sample. This notebook therefore prepares a transparent, dashboard-ready company identifier layer. It does not calculate event-study returns, does not download full historical prices, and does not build the Streamlit dashboard. Final returns and market-adjusted returns are postponed to later notebooks after identifiers have been verified.

The final academic submission is the Streamlit dashboard. Notebook 02 supports that dashboard by producing stable mapping artifacts that the dashboard and downstream notebooks can load directly.

## 1. Imports and Configuration

The notebook uses standard Python libraries and keeps online mapping checks disabled by default. Optional packages such as `yfinance` may be used later for lightweight validation, but this notebook must run without them.

In [17]:
import json
import re
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
import requests

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
except Exception:
    yf = None
    YFINANCE_AVAILABLE = False

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
DATA_INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
DATA_OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
SOURCE_TEST_DIR = CACHE_DIR / "source_tests"
MAPPING_CACHE_DIR = CACHE_DIR / "mapping"

for folder in [DATA_INTERIM_DIR, DATA_OUTPUT_DIR, CACHE_DIR, SOURCE_TEST_DIR, MAPPING_CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

ENABLE_ONLINE_MAPPING_TESTS = False
REQUEST_TIMEOUT_SECONDS = 10
REQUEST_SLEEP_SECONDS = 1.0
USER_AGENT = "AuditorSwitchEventStudyNotebook02/0.1 academic identifier mapping check"

print(f"Project root: {PROJECT_ROOT}")
print(f"yfinance available: {YFINANCE_AVAILABLE}")

Project root: C:\Users\Guy\Desktop\Tene
yfinance available: False


## 2. Load Notebook 01 Outputs

Notebook 02 depends on the cleaned event table and ticker-mapping template created by Notebook 01. If these files are missing, Notebook 01 should be rerun before continuing.

In [18]:
clean_events_path = DATA_INTERIM_DIR / "clean_events_v01.csv"
ticker_template_path = DATA_INTERIM_DIR / "ticker_mapping_template_v01.csv"
source_feasibility_path = DATA_OUTPUT_DIR / "source_feasibility_report_v01.csv"

missing_inputs = [path for path in [clean_events_path, ticker_template_path] if not path.exists()]
if missing_inputs:
    missing_text = ", ".join(str(path.relative_to(PROJECT_ROOT)) for path in missing_inputs)
    raise FileNotFoundError(
        f"Required Notebook 01 output(s) missing: {missing_text}. "
        "Run Notebook 01 before running Notebook 02."
    )

clean_events_df = pd.read_csv(clean_events_path, encoding="utf-8-sig")
ticker_template_df = pd.read_csv(ticker_template_path, encoding="utf-8-sig")

if source_feasibility_path.exists():
    source_feasibility_df = pd.read_csv(source_feasibility_path, encoding="utf-8-sig")
else:
    source_feasibility_df = pd.DataFrame()
    warnings.warn("source_feasibility_report_v01.csv was not found; continuing without source-feasibility context.")

print(f"Event rows: {len(clean_events_df)}")
print(f"Unique companies: {clean_events_df['company_name'].nunique(dropna=True)}")
display(clean_events_df["switch_type"].value_counts(dropna=False).rename_axis("switch_type").reset_index(name="event_count"))
display(clean_events_df["price_extraction_eligibility"].value_counts(dropna=False).rename_axis("price_extraction_eligibility").reset_index(name="event_count"))

unique_company_names = (
    clean_events_df["company_name"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)
display(pd.DataFrame({"company_name": unique_company_names}))

Event rows: 50
Unique companies: 49


,switch_type,event_count
0,Small to Small,29
1,Big to Small,11
2,Small to Big,10


,price_extraction_eligibility,event_count
0,ELIGIBLE_FOR_PRICE_EXTRACTION,48
1,FUTURE_EVENT_DATE,2


,company_name
0,א. לוי השקעות
1,אבו מגורים
2,אגוד הנפקות
3,אוברסיז
4,אוטונומוס
5,איי ארגנטו
6,אימקו
7,אינרום בניה
8,"אלקטרה נדל""ן"
9,"אמיליה פיתוח )מ.עו.פ.( בע""מ"


## 3. Normalize Company Names

The original company name is preserved exactly as loaded from Notebook 01. Normalized fields are added only for search, joining, and manual review convenience. This distinction is important because the original source wording may be needed for auditability.

In [19]:
HEBREW_LEGAL_SUFFIX_PATTERNS = [
    r'\bבע["״]?מ\b',
    r'\bבעמ\b',
    r'\bבע"מ\b',
    r'\bבע״מ\b',
    r'\bבע"מ\.\b',
    r'\bבע״מ\.\b',
]

PUNCT_TRANSLATION = str.maketrans({
    "״": '"',
    "׳": "'",
    "’": "'",
    "‘": "'",
    "`": "'",
    "‐": "-",
    "‑": "-",
    "–": "-",
    "—": "-",
})


def normalize_hebrew_company_name(name):
    if pd.isna(name):
        return ""
    text = str(name).strip().translate(PUNCT_TRANSLATION)
    text = text.replace('\u200f', '').replace('\u200e', '')
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*-\s*", "-", text)
    return text.strip()


def remove_legal_suffixes(name):
    text = normalize_hebrew_company_name(name)
    for pattern in HEBREW_LEGAL_SUFFIX_PATTERNS:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r'[,".;:()\[\]{}]+$', "", text)
    return text.strip()


def company_search_key(name):
    text = remove_legal_suffixes(name).lower()
    text = re.sub(r"[\"'.,;:()\[\]{}]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def create_search_variants(name):
    normalized = normalize_hebrew_company_name(name)
    no_suffix = remove_legal_suffixes(name)
    variants = [normalized, no_suffix, company_search_key(name)]
    variants = [variant for variant in variants if variant]
    return list(dict.fromkeys(variants))


name_normalization_preview_df = pd.DataFrame({
    "company_name_original": unique_company_names,
})
name_normalization_preview_df["company_name_normalized"] = name_normalization_preview_df["company_name_original"].apply(normalize_hebrew_company_name)
name_normalization_preview_df["company_name_search_key"] = name_normalization_preview_df["company_name_original"].apply(company_search_key)
name_normalization_preview_df["search_variants"] = name_normalization_preview_df["company_name_original"].apply(create_search_variants)
name_normalization_preview_df.head(10)

,company_name_original,company_name_normalized,company_name_search_key,search_variants
0,א. לוי השקעות,א. לוי השקעות,א לוי השקעות,"[א. לוי השקעות, א לוי השקעות]"
1,אבו מגורים,אבו מגורים,אבו מגורים,[אבו מגורים]
2,אגוד הנפקות,אגוד הנפקות,אגוד הנפקות,[אגוד הנפקות]
3,אוברסיז,אוברסיז,אוברסיז,[אוברסיז]
4,אוטונומוס,אוטונומוס,אוטונומוס,[אוטונומוס]
5,איי ארגנטו,איי ארגנטו,איי ארגנטו,[איי ארגנטו]
6,אימקו,אימקו,אימקו,[אימקו]
7,אינרום בניה,אינרום בניה,אינרום בניה,[אינרום בניה]
8,"אלקטרה נדל""ן","אלקטרה נדל""ן",אלקטרה נדל ן,"[אלקטרה נדל""ן, אלקטרה נדל ן]"
9,"אמיליה פיתוח )מ.עו.פ.( בע""מ","אמיליה פיתוח )מ.עו.פ.( בע""מ",אמיליה פיתוח מ עו פ,"[אמיליה פיתוח )מ.עו.פ.( בע""מ, אמיליה פיתוח )מ...."


## 4. Build Mapping Worktable

The worktable has one row per unique company. All identifier fields begin empty unless supplied by Notebook 01 templates or a prior manual mapping file. By default, each row requires manual review because Hebrew company names alone are not sufficient to infer verified market identifiers.

In [20]:
mapping_df = pd.DataFrame({"company_name_original": unique_company_names})
mapping_df["company_name_normalized"] = mapping_df["company_name_original"].apply(normalize_hebrew_company_name)
mapping_df["company_name_search_key"] = mapping_df["company_name_original"].apply(company_search_key)

identifier_columns = [
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "maya_company_url",
    "tase_company_url",
    "bizportal_url",
    "investing_url",
    "globes_search_url",
]
for column in identifier_columns:
    mapping_df[column] = ""

mapping_df["mapping_status"] = "PENDING"
mapping_df["mapping_confidence"] = "UNKNOWN"
mapping_df["mapping_method"] = ""
mapping_df["mapping_notes"] = ""
mapping_df["needs_manual_review"] = True

mapping_df.head()

,company_name_original,company_name_normalized,company_name_search_key,yahoo_ticker,google_finance_symbol,tase_security_id,isin,maya_company_url,tase_company_url,bizportal_url,investing_url,globes_search_url,mapping_status,mapping_confidence,mapping_method,mapping_notes,needs_manual_review
0,א. לוי השקעות,א. לוי השקעות,א לוי השקעות,,,,,,,,,,PENDING,UNKNOWN,,,True
1,אבו מגורים,אבו מגורים,אבו מגורים,,,,,,,,,,PENDING,UNKNOWN,,,True
2,אגוד הנפקות,אגוד הנפקות,אגוד הנפקות,,,,,,,,,,PENDING,UNKNOWN,,,True
3,אוברסיז,אוברסיז,אוברסיז,,,,,,,,,,PENDING,UNKNOWN,,,True
4,אוטונומוס,אוטונומוס,אוטונומוס,,,,,,,,,,PENDING,UNKNOWN,,,True


## 5. Manual Mapping Support

Manual mapping entries are treated as the most trusted source in this notebook. If a prior mapping workbook exists, its identifiers are merged into the current worktable. A row is marked as manually mapped only when at least one usable identifier is present.

In [21]:
MANUAL_CANDIDATE_FILES = [
    DATA_INTERIM_DIR / "company_identifier_mapping_manual.xlsx",
    DATA_INTERIM_DIR / "company_identifier_mapping_v01.xlsx",
]

manual_files_loaded = []
manual_identifier_columns = [
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "maya_company_url",
    "tase_company_url",
    "bizportal_url",
    "investing_url",
    "globes_search_url",
    "mapping_notes",
]


def clean_identifier_value(value):
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.lower() in {"nan", "none", "null"}:
        return ""
    return text


def has_any_usable_identifier(row):
    return any(clean_identifier_value(row.get(column, "")) for column in ["yahoo_ticker", "google_finance_symbol", "tase_security_id", "isin"])


def merge_manual_identifier_frame(mapping_table, manual_df, source_name):
    if "company_name_original" not in manual_df.columns:
        if "company_name" in manual_df.columns:
            manual_df = manual_df.rename(columns={"company_name": "company_name_original"})
        else:
            warnings.warn(f"Skipping {source_name}: no company_name_original/company_name column.")
            return mapping_table, False

    manual_df = manual_df.copy()
    manual_df["company_name_search_key"] = manual_df["company_name_original"].apply(company_search_key)
    manual_subset_columns = ["company_name_search_key"] + [col for col in manual_identifier_columns if col in manual_df.columns]
    manual_subset = manual_df[manual_subset_columns].copy()
    manual_subset = manual_subset.drop_duplicates("company_name_search_key", keep="last")

    mapping_table = mapping_table.merge(
        manual_subset,
        on="company_name_search_key",
        how="left",
        suffixes=("", "__manual"),
    )
    for column in manual_identifier_columns:
        manual_column = f"{column}__manual"
        if manual_column in mapping_table.columns:
            manual_values = mapping_table[manual_column].apply(clean_identifier_value)
            existing_values = mapping_table[column].apply(clean_identifier_value) if column in mapping_table.columns else ""
            mapping_table[column] = np.where(manual_values.ne(""), manual_values, existing_values)
            mapping_table = mapping_table.drop(columns=[manual_column])
    return mapping_table, True


# The Notebook 01 ticker template is loaded as a seed. If the user has filled
# identifiers into that template, those entries are treated as manual inputs.
mapping_df, loaded_template_seed = merge_manual_identifier_frame(
    mapping_df,
    ticker_template_df,
    ticker_template_path.name,
)
if loaded_template_seed:
    manual_files_loaded.append(ticker_template_path.name)

for manual_path in MANUAL_CANDIDATE_FILES:
    if not manual_path.exists():
        continue
    manual_df = pd.read_excel(manual_path, dtype=object)
    mapping_df, loaded_manual = merge_manual_identifier_frame(mapping_df, manual_df, manual_path.name)
    if loaded_manual:
        manual_files_loaded.append(manual_path.name)

manual_identifier_mask = mapping_df.apply(has_any_usable_identifier, axis=1)
mapping_df.loc[manual_identifier_mask, "mapping_method"] = "MANUAL"
mapping_df.loc[manual_identifier_mask, "mapping_status"] = "MAPPED_MANUALLY"
mapping_df.loc[manual_identifier_mask, "mapping_confidence"] = "HIGH"
mapping_df.loc[manual_identifier_mask, "needs_manual_review"] = False

print(f"Manual/template mapping files loaded: {manual_files_loaded if manual_files_loaded else 'None'}")
display(mapping_df.head())

Manual/template mapping files loaded: ['ticker_mapping_template_v01.csv', 'company_identifier_mapping_manual.xlsx', 'company_identifier_mapping_v01.xlsx']


,company_name_original,company_name_normalized,company_name_search_key,yahoo_ticker,google_finance_symbol,tase_security_id,isin,maya_company_url,tase_company_url,bizportal_url,investing_url,globes_search_url,mapping_status,mapping_confidence,mapping_method,mapping_notes,needs_manual_review
0,א. לוי השקעות,א. לוי השקעות,א לוי השקעות,LEVA.TA,TLV:LEVA,,,,,,,https://www.google.com/search?q=%D7%90.+%D7%9C...,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False
1,אבו מגורים,אבו מגורים,אבו מגורים,ABOU.TA,TLV:ABOU,01175819,,,,,,https://www.google.com/search?q=%D7%90%D7%91%D...,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False
2,אגוד הנפקות,אגוד הנפקות,אגוד הנפקות,,,,,,,,,https://www.google.com/search?q=%D7%90%D7%92%D...,PENDING,UNKNOWN,,Identifier verification required before price ...,True
3,אוברסיז,אוברסיז,אוברסיז,OVRS.TA,TLV:OVRS,01139617,,,,,,https://www.google.com/search?q=%D7%90%D7%95%D...,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False
4,אוטונומוס,אוטונומוס,אוטונומוס,AGRD.TA,TLV:AGRD,01083419,,,,,,https://www.google.com/search?q=%D7%90%D7%95%D...,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False


## 6. Generate Candidate Search URLs

The URLs below are safe reference links for manual verification and future source resolution. The notebook does not scrape search result pages. Search queries are encoded with `urllib.parse.quote_plus`.

In [22]:
def build_query(company_name, suffix):
    return f"{company_name} {suffix}".strip()


def google_search_url(query):
    return "https://www.google.com/search?q=" + quote_plus(query)


def source_search_url(base_url, query_param_name, query):
    return f"{base_url}?{query_param_name}={quote_plus(query)}"


mapping_df["google_search_query"] = mapping_df["company_name_original"].apply(lambda name: build_query(name, "מניה מאיה"))
mapping_df["google_search_url"] = mapping_df["google_search_query"].apply(google_search_url)
mapping_df["maya_search_url"] = mapping_df["company_name_original"].apply(lambda name: google_search_url(build_query(name, "site:maya.tase.co.il")))
mapping_df["tase_search_url"] = mapping_df["company_name_original"].apply(lambda name: google_search_url(build_query(name, "בורסה תל אביב נייר ערך")))
mapping_df["bizportal_search_url"] = mapping_df["company_name_original"].apply(lambda name: google_search_url(build_query(name, "ביזפורטל מניה")))
mapping_df["investing_search_url"] = mapping_df["company_name_original"].apply(lambda name: google_search_url(build_query(name, "Investing.com")))
mapping_df["globes_search_url"] = mapping_df["company_name_original"].apply(lambda name: google_search_url(build_query(name, "גלובס מניה")))

mapping_df[[
    "company_name_original",
    "google_search_url",
    "maya_search_url",
    "tase_search_url",
    "bizportal_search_url",
    "investing_search_url",
    "globes_search_url",
]].head()

,company_name_original,google_search_url,maya_search_url,tase_search_url,bizportal_search_url,investing_search_url,globes_search_url
0,א. לוי השקעות,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...,https://www.google.com/search?q=%D7%90.+%D7%9C...
1,אבו מגורים,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...,https://www.google.com/search?q=%D7%90%D7%91%D...
2,אגוד הנפקות,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...
3,אוברסיז,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...
4,אוטונומוס,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...,https://www.google.com/search?q=%D7%90%D7%95%D...


## 7. Optional Lightweight Online Resolution

Online mapping tests are disabled by default. If enabled, this cell performs only polite accessibility checks for a very small set of reference URLs. It does not scrape full pages, does not use Selenium, does not bypass CAPTCHAs, and does not extract historical prices. Failed requests are recorded rather than allowed to crash the notebook.

In [23]:
def test_mapping_url(company_name, source_name, url):
    record = {
        "company_name_original": company_name,
        "source_name": source_name,
        "url": url,
        "http_status_code": None,
        "accessible_boolean": False,
        "error_message": "",
        "tested_at": datetime.now(timezone.utc).isoformat(),
    }
    try:
        response = requests.get(
            url,
            timeout=REQUEST_TIMEOUT_SECONDS,
            headers={"User-Agent": USER_AGENT},
        )
        record["http_status_code"] = int(response.status_code)
        record["accessible_boolean"] = bool(200 <= response.status_code < 400)
    except requests.RequestException as exc:
        record["error_message"] = f"INTERNET_UNAVAILABLE_OR_BLOCKED: {type(exc).__name__}: {exc}"
    return record


access_records = []
if ENABLE_ONLINE_MAPPING_TESTS:
    sample_for_access_tests = mapping_df.head(5)
    for _, row in sample_for_access_tests.iterrows():
        for source_name, url_column in [
            ("Google Search", "google_search_url"),
            ("Maya Search", "maya_search_url"),
            ("TASE Search", "tase_search_url"),
        ]:
            access_records.append(test_mapping_url(row["company_name_original"], source_name, row[url_column]))
            time.sleep(REQUEST_SLEEP_SECONDS)
else:
    access_records.append({
        "company_name_original": "",
        "source_name": "ALL",
        "url": "",
        "http_status_code": None,
        "accessible_boolean": False,
        "error_message": "NOT_RUN_OPTION_DISABLED",
        "tested_at": datetime.now(timezone.utc).isoformat(),
    })

company_mapping_source_access_report_df = pd.DataFrame(access_records)
access_report_path = DATA_OUTPUT_DIR / "company_mapping_source_access_report_v01.csv"
company_mapping_source_access_report_df.to_csv(access_report_path, index=False, encoding="utf-8-sig")
company_mapping_source_access_report_df

,company_name_original,source_name,url,http_status_code,accessible_boolean,error_message,tested_at
0,,ALL,,None,False,NOT_RUN_OPTION_DISABLED,2026-05-27T11:52:26.796332+00:00


## 8. Yahoo and Google Finance Candidate Generation

Yahoo Finance tickers for Israeli equities often use the `.TA` suffix, and Google Sheets `GOOGLEFINANCE` symbols often use the `TLV:` prefix. However, these symbols cannot be reliably inferred from Hebrew company names alone. This notebook therefore preserves manually supplied tickers and creates candidates only when there is a clear English all-capital symbol already present in the company name.

In [24]:
def clear_english_symbol_from_name(name):
    if pd.isna(name):
        return ""
    text = str(name)
    tokens = re.findall(r"\b[A-Z]{2,6}\b", text)
    blocked = {"LTD", "INC", "PLC", "THE"}
    candidates = [token for token in tokens if token not in blocked]
    return candidates[0] if len(candidates) == 1 else ""


mapping_df["english_symbol_candidate"] = mapping_df["company_name_original"].apply(clear_english_symbol_from_name)
mapping_df["yahoo_ticker_candidate"] = mapping_df["english_symbol_candidate"].apply(lambda value: f"{value}.TA" if value else "")
mapping_df["google_finance_symbol_candidate"] = mapping_df["english_symbol_candidate"].apply(lambda value: f"TLV:{value}" if value else "")

# Do not overwrite verified/manual identifiers with unverified candidates.
mapping_df[[
    "company_name_original",
    "yahoo_ticker",
    "yahoo_ticker_candidate",
    "google_finance_symbol",
    "google_finance_symbol_candidate",
]].head(10)

,company_name_original,yahoo_ticker,yahoo_ticker_candidate,google_finance_symbol,google_finance_symbol_candidate
0,א. לוי השקעות,LEVA.TA,,TLV:LEVA,
1,אבו מגורים,ABOU.TA,,TLV:ABOU,
2,אגוד הנפקות,,,,
3,אוברסיז,OVRS.TA,,TLV:OVRS,
4,אוטונומוס,AGRD.TA,,TLV:AGRD,
5,איי ארגנטו,IARG.TA,,TLV:IARG,
6,אימקו,IMCO.TA,,TLV:IMCO,
7,אינרום בניה,INRM.TA,,TLV:INRM,
8,"אלקטרה נדל""ן",ELCRE.TA,,TLV:ELCRE,
9,"אמיליה פיתוח )מ.עו.פ.( בע""מ",EMDV.TA,,TLV:EMDV,


## 9. Mapping Status Logic

Mapping status remains conservative. Rows with manually supplied usable identifiers are treated as high-confidence manual mappings. Rows with identifiers from non-manual sources are partial unless independently verified. Rows without identifiers remain pending manual review.

In [25]:
def compute_mapping_status(row):
    company = clean_identifier_value(row.get("company_name_original", ""))
    if not company:
        return "INVALID_COMPANY_NAME", "UNKNOWN", True

    has_identifier = has_any_usable_identifier(row)
    method = clean_identifier_value(row.get("mapping_method", ""))

    if method == "MANUAL" and has_identifier:
        return "MAPPED_MANUALLY", "HIGH", False
    if method == "AUTOMATIC_VERIFIED" and has_identifier:
        return "MAPPED_AUTOMATICALLY_VERIFIED", "HIGH", False
    if has_identifier:
        return "PARTIAL_MAPPING", "MEDIUM", True
    return "PENDING_MANUAL_REVIEW", "UNKNOWN", True


status_parts = mapping_df.apply(compute_mapping_status, axis=1, result_type="expand")
mapping_df["mapping_status"] = status_parts[0]
mapping_df["mapping_confidence"] = status_parts[1]
mapping_df["needs_manual_review"] = status_parts[2].astype(bool)
mapping_df.loc[mapping_df["mapping_status"].eq("PENDING_MANUAL_REVIEW") & mapping_df["mapping_notes"].eq(""), "mapping_notes"] = (
    "Identifier verification required before price extraction."
)

mapping_df["mapping_status"].value_counts(dropna=False).rename_axis("mapping_status").reset_index(name="company_count")

,mapping_status,company_count
0,MAPPED_MANUALLY,48
1,PENDING_MANUAL_REVIEW,1


## 10. Create Dashboard-Ready Mapping Table

The final company mapping table uses predictable column names and one row per unique company. It is designed for downstream notebooks and the final Streamlit dashboard.

In [26]:
FINAL_MAPPING_COLUMNS = [
    "company_name_original",
    "company_name_normalized",
    "company_name_search_key",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "yahoo_ticker_candidate",
    "google_finance_symbol_candidate",
    "google_search_url",
    "maya_search_url",
    "tase_search_url",
    "bizportal_search_url",
    "investing_search_url",
    "globes_search_url",
    "maya_company_url",
    "tase_company_url",
    "bizportal_url",
    "investing_url",
    "mapping_status",
    "mapping_confidence",
    "mapping_method",
    "mapping_notes",
    "needs_manual_review",
]

missing_mapping_columns = [column for column in FINAL_MAPPING_COLUMNS if column not in mapping_df.columns]
if missing_mapping_columns:
    raise ValueError(f"Mapping table is missing required columns: {missing_mapping_columns}")

company_identifier_mapping_df = (
    mapping_df[FINAL_MAPPING_COLUMNS]
    .drop_duplicates("company_name_search_key", keep="first")
    .sort_values("company_name_original")
    .reset_index(drop=True)
)

mapping_csv_path = DATA_INTERIM_DIR / "company_identifier_mapping_v01.csv"
mapping_xlsx_path = DATA_INTERIM_DIR / "company_identifier_mapping_v01.xlsx"
company_identifier_mapping_df.to_csv(mapping_csv_path, index=False, encoding="utf-8-sig")
company_identifier_mapping_df.to_excel(mapping_xlsx_path, index=False)

company_identifier_mapping_df.head()

,company_name_original,company_name_normalized,company_name_search_key,yahoo_ticker,google_finance_symbol,tase_security_id,isin,yahoo_ticker_candidate,google_finance_symbol_candidate,google_search_url,...,globes_search_url,maya_company_url,tase_company_url,bizportal_url,investing_url,mapping_status,mapping_confidence,mapping_method,mapping_notes,needs_manual_review
0,א. לוי השקעות,א. לוי השקעות,א לוי השקעות,LEVA.TA,TLV:LEVA,,,,,https://www.google.com/search?q=%D7%90.+%D7%9C...,...,https://www.google.com/search?q=%D7%90.+%D7%9C...,,,,,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False
1,אבו מגורים,אבו מגורים,אבו מגורים,ABOU.TA,TLV:ABOU,01175819,,,,https://www.google.com/search?q=%D7%90%D7%91%D...,...,https://www.google.com/search?q=%D7%90%D7%91%D...,,,,,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False
2,אגוד הנפקות,אגוד הנפקות,אגוד הנפקות,,,,,,,https://www.google.com/search?q=%D7%90%D7%92%D...,...,https://www.google.com/search?q=%D7%90%D7%92%D...,,,,,PENDING_MANUAL_REVIEW,UNKNOWN,,Identifier verification required before price ...,True
3,אוברסיז,אוברסיז,אוברסיז,OVRS.TA,TLV:OVRS,01139617,,,,https://www.google.com/search?q=%D7%90%D7%95%D...,...,https://www.google.com/search?q=%D7%90%D7%95%D...,,,,,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False
4,אוטונומוס,אוטונומוס,אוטונומוס,AGRD.TA,TLV:AGRD,01083419,,,,https://www.google.com/search?q=%D7%90%D7%95%D...,...,https://www.google.com/search?q=%D7%90%D7%95%D...,,,,,MAPPED_MANUALLY,HIGH,MANUAL,Identifier verification required before price ...,False


## 11. Create Manual Review File

Rows requiring manual review are exported to a separate workbook designed for direct editing. The workbook keeps source/search URLs next to the identifier fields so manual verification decisions can be documented.

In [27]:
manual_review_columns = [
    "company_name_original",
    "company_name_normalized",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "google_search_url",
    "maya_search_url",
    "tase_search_url",
    "bizportal_search_url",
    "investing_search_url",
    "globes_search_url",
    "mapping_notes",
]

manual_review_df = company_identifier_mapping_df.loc[
    company_identifier_mapping_df["needs_manual_review"], manual_review_columns
].copy()

manual_review_path = DATA_INTERIM_DIR / "company_identifier_mapping_manual_review_v01.xlsx"
manual_review_df.to_excel(manual_review_path, index=False)

try:
    from openpyxl import load_workbook

    workbook = load_workbook(manual_review_path)
    worksheet = workbook.active
    worksheet.freeze_panes = "A2"
    for column_cells in worksheet.columns:
        max_length = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
        worksheet.column_dimensions[column_cells[0].column_letter].width = min(max(max_length + 2, 14), 70)
    workbook.save(manual_review_path)
except Exception as exc:
    warnings.warn(f"Manual review workbook was created, but formatting could not be applied: {exc}")

manual_review_df.head()

,company_name_original,company_name_normalized,yahoo_ticker,google_finance_symbol,tase_security_id,isin,google_search_url,maya_search_url,tase_search_url,bizportal_search_url,investing_search_url,globes_search_url,mapping_notes
2,אגוד הנפקות,אגוד הנפקות,,,,,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,https://www.google.com/search?q=%D7%90%D7%92%D...,Identifier verification required before price ...


## 12. Merge Mapping Back to Event-Level Data

The event-level table preserves every event from Notebook 01 and enriches each row with company-level identifiers and mapping status. No events are dropped.

In [28]:
event_mapping_columns = [
    "company_name_original",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "mapping_status",
    "mapping_confidence",
    "needs_manual_review",
]

event_level_mapping_df = company_identifier_mapping_df[event_mapping_columns].rename(
    columns={"company_name_original": "company_name"}
)

clean_events_with_identifiers_df = clean_events_df.merge(
    event_level_mapping_df,
    on="company_name",
    how="left",
    validate="many_to_one",
)

events_with_identifiers_csv_path = DATA_INTERIM_DIR / "clean_events_with_identifiers_v01.csv"
events_with_identifiers_xlsx_path = DATA_INTERIM_DIR / "clean_events_with_identifiers_v01.xlsx"
clean_events_with_identifiers_df.to_csv(events_with_identifiers_csv_path, index=False, encoding="utf-8-sig")
clean_events_with_identifiers_df.to_excel(events_with_identifiers_xlsx_path, index=False)

clean_events_with_identifiers_df[[
    "company_name",
    "event_date",
    "switch_type",
    "price_extraction_eligibility",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "mapping_status",
    "mapping_confidence",
    "needs_manual_review",
]].head(10)

,company_name,event_date,switch_type,price_extraction_eligibility,yahoo_ticker,google_finance_symbol,tase_security_id,isin,mapping_status,mapping_confidence,needs_manual_review
0,אבו מגורים,2025-11-16,Small to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,ABOU.TA,TLV:ABOU,01175819,,MAPPED_MANUALLY,HIGH,False
1,גלובל פיי-ש,2025-02-19,Small to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,GKL.TA,TLV:GKL,01141316,,MAPPED_MANUALLY,HIGH,False
2,"אמיליה פיתוח )מ.עו.פ.( בע""מ",2026-02-17,Small to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,EMDV.TA,TLV:EMDV,,,MAPPED_MANUALLY,HIGH,False
3,"שדה נדל""ן",2024-05-03,Small to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,SADE.TA,TLV:SADE,,IL0003410169,MAPPED_MANUALLY,HIGH,False
4,ספידווליו,2025-08-24,Big to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,SPDV.TA,TLV:SPDV,01178284,,MAPPED_MANUALLY,HIGH,False
5,הום ביוגז,2024-07-17,Big to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,HMGS.TA,TLV:HMGS,01172204,,MAPPED_MANUALLY,HIGH,False
6,טוגדר,2025-07-28,Small to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,TGTR.TA,TLV:TGTR,,,MAPPED_MANUALLY,HIGH,False
7,אספן גרופ,2026-05-24,Small to Small,ELIGIBLE_FOR_PRICE_EXTRACTION,ASGR.TA,TLV:ASGR,00313015,,MAPPED_MANUALLY,HIGH,False
8,ג׳נסל בע״מ,2026-07-05,Big to Small,FUTURE_EVENT_DATE,GNCL.TA,TLV:GNCL,01169689,,MAPPED_MANUALLY,HIGH,False
9,נופר אנרג׳י,2026-07-05,Small to Small,FUTURE_EVENT_DATE,NOFR.TA,TLV:NOFR,01170877,,MAPPED_MANUALLY,HIGH,False


## 13. Mapping Quality Report

The mapping-quality report summarizes readiness for later price extraction. A company is considered to have an identifier if at least one of Yahoo ticker, Google Finance symbol, TASE security identifier, or ISIN is present. A company is considered ready for price extraction only when the mapping is manually supplied or automatically verified.

In [29]:
status_counts = company_identifier_mapping_df["mapping_status"].value_counts(dropna=False)
any_identifier_mask = company_identifier_mapping_df.apply(has_any_usable_identifier, axis=1)
ready_statuses = ["MAPPED_MANUALLY", "MAPPED_AUTOMATICALLY_VERIFIED"]
ready_for_price_extraction_mask = company_identifier_mapping_df["mapping_status"].isin(ready_statuses)

mapping_quality_summary = {
    "number_of_unique_companies": int(len(company_identifier_mapping_df)),
    "number_mapped_manually": int(status_counts.get("MAPPED_MANUALLY", 0)),
    "number_mapped_automatically_verified": int(status_counts.get("MAPPED_AUTOMATICALLY_VERIFIED", 0)),
    "number_partial_mapping": int(status_counts.get("PARTIAL_MAPPING", 0)),
    "number_pending_manual_review": int(status_counts.get("PENDING_MANUAL_REVIEW", 0)),
    "number_no_identifier_found": int(status_counts.get("NO_IDENTIFIER_FOUND", 0)),
    "number_invalid_company_name": int(status_counts.get("INVALID_COMPANY_NAME", 0)),
    "percentage_with_any_identifier": round(float(any_identifier_mask.mean() * 100), 2) if len(company_identifier_mapping_df) else 0.0,
    "percentage_ready_for_price_extraction": round(float(ready_for_price_extraction_mask.mean() * 100), 2) if len(company_identifier_mapping_df) else 0.0,
}

mapping_quality_report_df = pd.DataFrame([
    {"metric": metric, "value": value}
    for metric, value in mapping_quality_summary.items()
])

mapping_quality_csv_path = DATA_OUTPUT_DIR / "mapping_quality_report_v01.csv"
mapping_quality_json_path = DATA_OUTPUT_DIR / "mapping_quality_report_v01.json"
mapping_quality_report_df.to_csv(mapping_quality_csv_path, index=False, encoding="utf-8-sig")
mapping_quality_json_path.write_text(
    json.dumps(mapping_quality_summary, ensure_ascii=False, indent=2, sort_keys=True),
    encoding="utf-8",
)

mapping_quality_report_df

,metric,value
0,number_of_unique_companies,49.00
1,number_mapped_manually,48.00
2,number_mapped_automatically_verified,0.00
3,number_partial_mapping,0.00
4,number_pending_manual_review,1.00
5,number_no_identifier_found,0.00
6,number_invalid_company_name,0.00
7,percentage_with_any_identifier,97.96
8,percentage_ready_for_price_extraction,97.96


## 14. Summary Tables

These summary tables show the current mapping status, the companies requiring manual review, a preview of the event-level merge, and the mapping-quality summary.

In [30]:
display(company_identifier_mapping_df["mapping_status"].value_counts(dropna=False).rename_axis("mapping_status").reset_index(name="company_count"))
display(company_identifier_mapping_df.loc[
    company_identifier_mapping_df["needs_manual_review"],
    ["company_name_original", "mapping_status", "mapping_confidence", "mapping_notes", "google_search_url"]
])
display(clean_events_with_identifiers_df.head())
display(mapping_quality_report_df)

,mapping_status,company_count
0,MAPPED_MANUALLY,48
1,PENDING_MANUAL_REVIEW,1


,company_name_original,mapping_status,mapping_confidence,mapping_notes,google_search_url
2,אגוד הנפקות,PENDING_MANUAL_REVIEW,UNKNOWN,Identifier verification required before price ...,https://www.google.com/search?q=%D7%90%D7%92%D...


,company_name,auditor_switch_date_raw,auditor_switch_date,auditor_switch_date_status,report_publication_date_raw,report_publication_date,report_publication_date_status,event_date_basis,event_date_raw,event_date,...,raw__סך הנכסים,raw__הון עצמי,raw__רווח,yahoo_ticker,google_finance_symbol,tase_security_id,isin,mapping_status,mapping_confidence,needs_manual_review
0,אבו מגורים,16/11/2025,2025-11-16,OK,19/3/2026,2026-03-19,OK,auditor_switch_date,16/11/2025,2025-11-16,...,7.745000e+08,2.406000e+08,4540000.0,ABOU.TA,TLV:ABOU,01175819,,MAPPED_MANUALLY,HIGH,False
1,גלובל פיי-ש,19/02/2025,2025-02-19,OK,31/3/2025,2025-03-31,OK,auditor_switch_date,19/02/2025,2025-02-19,...,1.800000e+07,1.110000e+07,510000.0,GKL.TA,TLV:GKL,01141316,,MAPPED_MANUALLY,HIGH,False
2,"אמיליה פיתוח )מ.עו.פ.( בע""מ",17/02/2026,2026-02-17,OK,30/3/2026,2026-03-30,OK,auditor_switch_date,17/02/2026,2026-02-17,...,4.340000e+09,1.570000e+09,151500000.0,EMDV.TA,TLV:EMDV,,,MAPPED_MANUALLY,HIGH,False
3,"שדה נדל""ן",2024-05-03 00:00:00,2024-05-03,OK,31/3/2024,2024-03-31,OK,auditor_switch_date,2024-05-03 00:00:00,2024-05-03,...,1.530000e+07,1.390000e+07,-14400000.0,SADE.TA,TLV:SADE,,IL0003410169,MAPPED_MANUALLY,HIGH,False
4,ספידווליו,24/08/2025,2025-08-24,OK,25/03/2026,2026-03-25,OK,auditor_switch_date,24/08/2025,2025-08-24,...,1.246000e+08,5.180000e+07,4860000.0,SPDV.TA,TLV:SPDV,01178284,,MAPPED_MANUALLY,HIGH,False


,metric,value
0,number_of_unique_companies,49.00
1,number_mapped_manually,48.00
2,number_mapped_automatically_verified,0.00
3,number_partial_mapping,0.00
4,number_pending_manual_review,1.00
5,number_no_identifier_found,0.00
6,number_invalid_company_name,0.00
7,percentage_with_any_identifier,97.96
8,percentage_ready_for_price_extraction,97.96


## 15. Validation Checks

These checks ensure the notebook did not silently lose companies or events and that the dashboard-ready artifacts were created with the required schema.

In [31]:
required_mapping_columns = [
    "company_name_original",
    "company_name_normalized",
    "company_name_search_key",
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "yahoo_ticker_candidate",
    "google_finance_symbol_candidate",
    "google_search_url",
    "maya_search_url",
    "tase_search_url",
    "bizportal_search_url",
    "investing_search_url",
    "globes_search_url",
    "mapping_status",
    "mapping_confidence",
    "mapping_method",
    "mapping_notes",
    "needs_manual_review",
]
required_event_identifier_columns = [
    "yahoo_ticker",
    "google_finance_symbol",
    "tase_security_id",
    "isin",
    "mapping_status",
    "mapping_confidence",
    "needs_manual_review",
]

if len(clean_events_with_identifiers_df) != len(clean_events_df):
    raise ValueError(
        f"Event-level row count changed after merge: before={len(clean_events_df)}, after={len(clean_events_with_identifiers_df)}"
    )

if len(company_identifier_mapping_df) != clean_events_df["company_name"].dropna().nunique():
    raise ValueError("company_identifier_mapping_v01 does not have one row per unique non-missing company.")

missing_mapping_required = [column for column in required_mapping_columns if column not in company_identifier_mapping_df.columns]
if missing_mapping_required:
    raise ValueError(f"Mapping output is missing required columns: {missing_mapping_required}")

missing_event_required = [column for column in required_event_identifier_columns if column not in clean_events_with_identifiers_df.columns]
if missing_event_required:
    raise ValueError(f"Event-level identifier output is missing required columns: {missing_event_required}")

source_companies = set(clean_events_df["company_name"].dropna().astype(str))
mapped_companies = set(company_identifier_mapping_df["company_name_original"].dropna().astype(str))
lost_companies = sorted(source_companies - mapped_companies)
if lost_companies:
    raise ValueError(f"Company names were lost during mapping: {lost_companies}")

if not manual_review_path.exists():
    raise FileNotFoundError(f"Manual review file was not created: {manual_review_path}")

validation_summary = {
    "event_rows_preserved": int(len(clean_events_with_identifiers_df)),
    "unique_companies_mapped": int(len(company_identifier_mapping_df)),
    "manual_review_file_exists": manual_review_path.exists(),
    "missing_mapping_required_columns": missing_mapping_required,
    "missing_event_required_columns": missing_event_required,
}
validation_summary

{'event_rows_preserved': 50,
 'unique_companies_mapped': 49,
 'manual_review_file_exists': True,
 'missing_mapping_required_columns': [],
 'missing_event_required_columns': []}

## Notebook 02 Conclusions

This notebook creates the company identifier mapping layer required before historical price extraction. The conclusion below is generated from the current run so it reflects the actual mapping status of the project artifacts.

In [32]:
from IPython.display import Markdown, display

unique_companies_processed = len(company_identifier_mapping_df)
companies_with_identifiers = int(any_identifier_mask.sum())
companies_requiring_review = int(company_identifier_mapping_df["needs_manual_review"].sum())

saved_files = [
    mapping_csv_path,
    mapping_xlsx_path,
    manual_review_path,
    events_with_identifiers_csv_path,
    events_with_identifiers_xlsx_path,
    mapping_quality_csv_path,
    mapping_quality_json_path,
    access_report_path,
]
saved_files_text = "\n".join(f"- `{path.relative_to(PROJECT_ROOT)}`" for path in saved_files)

conclusion = f"""
### Notebook 02 Conclusions

- Unique companies processed: **{unique_companies_processed}**.
- Companies currently containing at least one identifier: **{companies_with_identifiers}**.
- Companies requiring manual review: **{companies_requiring_review}**.

Saved files:
{saved_files_text}

Price extraction is postponed to Notebook 03 because identifiers must be manually verified before historical stock prices, TA-125 prices, event-window prices, stock returns, market returns, and market-adjusted returns are calculated.

This notebook supports the final Streamlit dashboard by producing a stable company identifier table and an event-level dataset enriched with mapping status. Mapping uncertainty remains visible and should be shown in the dashboard because missing or uncertain tickers affect the reliability of event-study results.
"""

display(Markdown(conclusion))


### Notebook 02 Conclusions

- Unique companies processed: **49**.
- Companies currently containing at least one identifier: **48**.
- Companies requiring manual review: **1**.

Saved files:
- `data\interim\company_identifier_mapping_v01.csv`
- `data\interim\company_identifier_mapping_v01.xlsx`
- `data\interim\company_identifier_mapping_manual_review_v01.xlsx`
- `data\interim\clean_events_with_identifiers_v01.csv`
- `data\interim\clean_events_with_identifiers_v01.xlsx`
- `data\output\mapping_quality_report_v01.csv`
- `data\output\mapping_quality_report_v01.json`
- `data\output\company_mapping_source_access_report_v01.csv`

Price extraction is postponed to Notebook 03 because identifiers must be manually verified before historical stock prices, TA-125 prices, event-window prices, stock returns, market returns, and market-adjusted returns are calculated.

This notebook supports the final Streamlit dashboard by producing a stable company identifier table and an event-level dataset enriched with mapping status. Mapping uncertainty remains visible and should be shown in the dashboard because missing or uncertain tickers affect the reliability of event-study results.
